# DS-03 — PTV GTFS Schedule

DS-03 supplies the *arrive* link of the access chain and the graph behind the journey
endpoints. GTFS is the only feed in the register that publishes accessibility as a
first-class standard field rather than as free text or an inferred token.

Three things this notebook has to settle:

1. **`wheelchair_boarding` and `wheelchair_accessible`.** Both use an explicit
   three-value enumeration — 0 unknown, 1 accessible, 2 not accessible — which means,
   uniquely among our sources, a published *no* is distinguishable from *no
   information*. That sets the ceiling on how confident the arrive link can ever be.

2. **`pathways.txt` and `levels.txt`.** These model station interiors: walkways,
   stairs, escalators, **lifts**, fare gates and floors. If the metropolitan train feed
   populates them, then lifts exist as addressable objects in the schedule — which is
   the precondition for anything downstream that wants to know when a lift is out.
   If they are empty stubs, that is worth knowing today rather than in three weeks.

3. **Station structure.** `location_type` and `parent_station` decide whether
   `wheelchair_boarding` attaches to a station or to a platform, and those are
   different claims.

The archive is nested: `gtfs.zip` contains one numbered directory per mode, each
holding its own `google_transit.zip`.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import profile_lib as pl

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

print("project root:", pl.PROJECT_ROOT)
print("raw zone:    ", pl.RAW_ROOT)


project root: C:\Users\nitin\Documents\Projects\Final_Project\SportAble
raw zone:     C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_raw


In [2]:
import io, zipfile

raw = pl.resolve("DS-03")
p = pl.Profile(raw, "PTV GTFS Schedule")
p.check("raw_integrity",
        "pass" if raw.sha_matches_manifest else ("info" if raw.sha_matches_manifest is None else "fail"),
        f"SHA-256 of the profiled object is {raw.sha256}", raw.sha256)
raw

RawObject(DS-03 dt=2026-08-31 gtfs.zip 288,398,506B sha=d492e2bc8def… [match])

In [3]:
inv = pl.zip_inventory(raw)
p.observe("outer_zip_members", inv)
pd.DataFrame(inv)

,name,size_bytes,compressed_bytes
0,5/google_transit.zip,136564855,135611402
1,11/google_transit.zip,168623,155998
2,10/google_transit.zip,2036038,2027887
3,1/google_transit.zip,57442570,55568581
4,3/google_transit.zip,8143942,5951782
5,2/google_transit.zip,25480261,24048386
6,4/google_transit.zip,71660943,60596221
7,6/google_transit.zip,4490824,4437295


## Feed structure

PTV numbers its mode directories. The numbering is the publisher's, taken from its own
documentation, and is recorded here rather than inferred from route types.

In [4]:
PTV_MODES = {
    "1": "Regional train", "2": "Metropolitan train", "3": "Metropolitan tram",
    "4": "Metropolitan bus", "5": "Regional coach", "6": "Regional bus",
    "7": "TeleBus", "8": "Night bus", "10": "Interstate train", "11": "SkyBus",
}

outer = zipfile.ZipFile(raw.path)
inner_names = sorted(
    (n for n in outer.namelist() if n.lower().endswith("google_transit.zip")),
    key=lambda n: int(n.split("/")[0]) if n.split("/")[0].isdigit() else 99,
)
p.observe("inner_feed_count", len(inner_names))
p.check("feed_structure", "pass" if inner_names else "fail",
        f"{len(inner_names)} per-mode google_transit.zip members found inside the outer archive",
        inner_names)

unknown_modes = [n.split("/")[0] for n in inner_names if n.split("/")[0] not in PTV_MODES]
p.check("mode_numbering", "pass" if not unknown_modes else "warn",
        "every mode directory maps to a documented PTV mode"
        if not unknown_modes else
        f"undocumented mode directories present: {unknown_modes} — resolve against the publisher's "
        "documentation before loading, do not guess from route_type",
        unknown_modes)
inner_names

['1/google_transit.zip',
 '2/google_transit.zip',
 '3/google_transit.zip',
 '4/google_transit.zip',
 '5/google_transit.zip',
 '6/google_transit.zip',
 '10/google_transit.zip',
 '11/google_transit.zip']

In [5]:
# The outer archive is large, so each inner zip is decompressed once and reused.
# Reading it per member would decompress the whole thing on every call.
_cache: dict[str, zipfile.ZipFile] = {}

def inner_zip(name: str) -> zipfile.ZipFile:
    if name not in _cache:
        with outer.open(name) as fh:
            _cache[name] = zipfile.ZipFile(io.BytesIO(fh.read()))
    return _cache[name]

def read_inner(name: str, member: str, **kw):
    z = inner_zip(name)
    if member not in z.namelist():
        return None
    with z.open(member) as f:
        return pd.read_csv(f, low_memory=False, **kw)

structure = {}
for n in inner_names:
    mode_id = n.split("/")[0]
    structure[n] = {
        "mode_id": mode_id,
        "mode": PTV_MODES.get(mode_id, f"undocumented ({mode_id})"),
        "members": inner_zip(n).namelist(),
    }
p.observe("inner_feed_structure",
          {k: {"mode": v["mode"], "members": v["members"]} for k, v in structure.items()})
pd.DataFrame([{"mode_id": v["mode_id"], "mode": v["mode"], "files": len(v["members"])}
              for v in structure.values()])

,mode_id,mode,files
0,1,Regional train,11
1,2,Metropolitan train,11
2,3,Metropolitan tram,11
3,4,Metropolitan bus,11
4,5,Regional coach,11
5,6,Regional bus,11
6,10,Interstate train,11
7,11,SkyBus,11


## Required and optional files

A mode feed missing a required GTFS file is a load failure for that mode, not a silent
partial graph. The optional files are recorded separately because their *absence* is
itself a finding: no `pathways.txt` means no station interior, and no `feed_info.txt`
means the feed does not state its own validity window.

In [6]:
REQUIRED = ["agency.txt", "stops.txt", "routes.txt", "trips.txt", "stop_times.txt"]
OPTIONAL = ["feed_info.txt", "pathways.txt", "levels.txt", "transfers.txt",
            "calendar.txt", "calendar_dates.txt"]

rows = []
for n, meta in structure.items():
    row = {"mode": meta["mode"]}
    row["missing_required"] = [f for f in REQUIRED if f not in meta["members"]] or None
    for f in OPTIONAL:
        row[f] = f in meta["members"]
    rows.append(row)
files_df = pd.DataFrame(rows)
p.observe("gtfs_file_presence", files_df)

any_missing = files_df["missing_required"].notna().any()
p.check("gtfs_required_files", "fail" if any_missing else "pass",
        "every mode feed contains the required GTFS files" if not any_missing
        else "at least one mode feed is missing a required GTFS file",
        files_df[["mode", "missing_required"]].to_dict("records"))

no_feed_info = files_df.loc[~files_df["feed_info.txt"], "mode"].tolist()
p.check("feed_info_present", "pass" if not no_feed_info else "warn",
        "every mode feed ships feed_info.txt" if not no_feed_info
        else f"{len(no_feed_info)} mode feeds ship no feed_info.txt, so the feed does not state its "
             "own validity window — currency has to be derived from calendar.txt instead",
        no_feed_info)
if no_feed_info:
    p.contract("Where feed_info.txt is absent, derive feed currency from the calendar.txt date range and record which method was used per mode.")
files_df

,mode,missing_required,feed_info.txt,pathways.txt,levels.txt,transfers.txt,calendar.txt,calendar_dates.txt
0,Regional train,None,False,True,True,True,True,True
1,Metropolitan train,None,False,True,True,True,True,True
2,Metropolitan tram,None,False,True,True,True,True,True
3,Metropolitan bus,None,False,True,True,True,True,True
4,Regional coach,None,False,True,True,True,True,True
5,Regional bus,None,False,True,True,True,True,True
6,Interstate train,None,False,True,True,True,True,True
7,SkyBus,None,False,True,True,True,True,True


## Stops and `wheelchair_boarding`

GTFS defines:

| value | meaning |
|---|---|
| 0 or empty | no accessibility information for the stop |
| 1 | some vehicles at this stop can be boarded by a rider in a wheelchair |
| 2 | wheelchair boarding is not possible at this stop |

The third value is what makes this feed different from every other source in the
register.

In [7]:
WB_LABELS = {0: "0_no_information", 1: "1_accessible", 2: "2_not_accessible"}

frames = []
for n, meta in structure.items():
    s = read_inner(n, "stops.txt")
    if s is None:
        continue
    s = s.copy()
    s["mode"] = meta["mode"]
    s["mode_id"] = meta["mode_id"]
    frames.append(s)
stops = pd.concat(frames, ignore_index=True)

p.observe("stops_rows_all_modes", int(len(stops)))
p.observe("stops_unique_stop_id", int(stops["stop_id"].nunique()))
dup = int(len(stops) - stops["stop_id"].nunique())
p.check("stop_id_uniqueness", "pass" if dup == 0 else "warn",
        "stop_id is unique across every mode feed" if dup == 0
        else f"{dup:,} stop_id values appear in more than one mode feed — the load key must be "
             "(mode_id, stop_id), not stop_id alone",
        dup)
if dup:
    p.contract("Key stops on (mode_id, stop_id). A bare stop_id is not unique across PTV mode feeds.")
print(f"{len(stops):,} stop rows, {stops['stop_id'].nunique():,} distinct stop_id")
stops.head(3)

31,980 stop rows, 31,015 distinct stop_id


,stop_id,stop_name,stop_lat,stop_lon,stop_url,location_type,parent_station,wheelchair_boarding,level_id,mode,mode_id,platform_code,stop_code
0,11212,Flinders Street Station,-37.818095,144.966266,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,Regional train,1,NaN,NaN
1,11213,Flinders Street Station,-37.818144,144.966492,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,Regional train,1,NaN,NaN
2,11214,Flinders Street Station,-37.818198,144.966524,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,Regional train,1,NaN,NaN


In [8]:
if "wheelchair_boarding" not in stops.columns:
    p.check("wheelchair_boarding_present", "fail",
            "stops.txt does not carry wheelchair_boarding — the arrive link has no source", None)
else:
    stops["wb"] = (pd.to_numeric(stops["wheelchair_boarding"], errors="coerce")
                     .fillna(0).astype(int).map(WB_LABELS).fillna("other"))
    dist = stops["wb"].value_counts()
    p.observe("wheelchair_boarding_distribution_all_modes", dist)
    p.check("wheelchair_boarding_present", "pass",
            "stops.txt carries wheelchair_boarding with an explicit unknown value, so a published "
            "'not accessible' is distinguishable from 'not recorded'", dist)
    display(dist)

wb
0_no_information    29771
1_accessible         2198
2_not_accessible       11
Name: count, dtype: int64

### Resolving parent-station inheritance first

Under the GTFS spec a child stop with a blank `wheelchair_boarding` inherits its
parent station's value. Counting blanks before resolving that inheritance reports
stops as unknown when the station has already answered for them, so the resolution
has to happen before any coverage figure is computed, not after.

In [9]:
# Resolve inheritance before any distribution is reported.
raw_wb = pd.to_numeric(stops["wheelchair_boarding"], errors="coerce").fillna(0).astype(int)
stops["wb_raw"] = raw_wb.map(WB_LABELS).fillna("other")

has_parent = (
    stops["parent_station"].notna() & stops["parent_station"].astype(str).str.strip().ne("")
    if "parent_station" in stops.columns else pd.Series(False, index=stops.index)
)

parent_wb = {}
if "parent_station" in stops.columns and "location_type" in stops.columns:
    lt = pd.to_numeric(stops["location_type"], errors="coerce").fillna(0).astype(int)
    stations = stops[lt == 1]
    parent_wb = dict(zip(stations["stop_id"].astype(str), raw_wb[stations.index]))

inherited = stops["parent_station"].astype(str).map(parent_wb) if "parent_station" in stops.columns else None
resolved = raw_wb.copy()
if inherited is not None:
    take = has_parent & raw_wb.eq(0) & inherited.notna() & inherited.ne(0)
    resolved = resolved.mask(take, inherited)
    n_resolved = int(take.sum())
else:
    n_resolved = 0

stops["wb"] = resolved.astype(int).map(WB_LABELS).fillna("other")

p.observe("stops_resolved_by_parent_inheritance", n_resolved)
p.check("parent_inheritance_applied", "info",
        f"{n_resolved:,} child stops inherited a wheelchair_boarding value from their parent "
        "station; every figure below is computed after this resolution, never before",
        n_resolved)
p.contract("Resolve wheelchair_boarding through parent_station before computing coverage. A blank on a parented child inherits from its station and is not an unknown.")
print(f"resolved by inheritance: {n_resolved:,}")
pd.DataFrame({"before": stops["wb_raw"].value_counts(), "after": stops["wb"].value_counts()})

resolved by inheritance: 1,340


,before,after
0_no_information,29771,28431
1_accessible,2198,3538
2_not_accessible,11,11


In [10]:
by_mode = stops.groupby(["mode", "wb"]).size().unstack(fill_value=0)
for c in WB_LABELS.values():
    if c not in by_mode.columns:
        by_mode[c] = 0
by_mode["total"] = by_mode[list(WB_LABELS.values())].sum(axis=1)
by_mode["pct_accessible"] = (100 * by_mode["1_accessible"] / by_mode["total"]).round(1)
by_mode["pct_no_information"] = (100 * by_mode["0_no_information"] / by_mode["total"]).round(1)
p.observe("wheelchair_boarding_by_mode", by_mode)
by_mode.sort_values("total", ascending=False)

wb,0_no_information,1_accessible,2_not_accessible,total,pct_accessible,pct_no_information
mode,,,,,,
Metropolitan bus,22578,0,0,22578,0.0,100.0
Regional bus,3324,0,0,3324,0.0,100.0
Metropolitan train,4,2846,9,2859,99.5,0.1
Metropolitan tram,1620,0,0,1620,0.0,100.0
Regional coach,871,0,0,871,0.0,100.0
Regional train,0,613,2,615,99.7,0.0
Interstate train,0,79,0,79,100.0,0.0
SkyBus,34,0,0,34,0.0,100.0


In [11]:
gm_stops = stops[
    stops["stop_lat"].between(pl.GM_BBOX["min_lat"], pl.GM_BBOX["max_lat"])
    & stops["stop_lon"].between(pl.GM_BBOX["min_lon"], pl.GM_BBOX["max_lon"])
].copy()
gm_dist = gm_stops["wb"].value_counts()
unknown_share = round(100 * gm_dist.get("0_no_information", 0) / len(gm_stops), 1)

p.observe("stops_in_gm_bbox", int(len(gm_stops)))
p.observe("wheelchair_boarding_distribution_gm", gm_dist)
p.observe("pct_stops_no_accessibility_information_gm", unknown_share)
p.check("arrive_link_coverage", "warn" if unknown_share > 20 else "pass",
        f"{unknown_share}% of Greater Melbourne stops publish no wheelchair boarding information; "
        f"{gm_dist.get('1_accessible', 0):,} are published accessible and "
        f"{gm_dist.get('2_not_accessible', 0):,} are published not accessible",
        {"distribution": gm_dist.to_dict(), "pct_unknown": unknown_share})
p.contract("Load wheelchair_boarding as a three-value enumeration. Never coalesce 0 into 2.")
p.limitation(
    f"DS-03 records no wheelchair boarding information for {unknown_share}% of Greater Melbourne "
    "stops. Those stops are shown as no published information, not as inaccessible."
)
gm_dist

wb
0_no_information    22895
1_accessible         3408
2_not_accessible       11
Name: count, dtype: int64

## Station structure

`location_type` 1 is a station, 0 or empty is a stop or platform, 2 is an entrance.
Where platforms sit under a parent station, `wheelchair_boarding` on the platform
inherits from the station when it is left blank — so a naive count treats an inherited
blank as unknown when the station has actually answered.

In [12]:
if "location_type" not in stops.columns:
    p.check("station_structure", "warn", "stops.txt carries no location_type — station and platform "
            "cannot be distinguished, and parent inheritance cannot be applied", None)
else:
    lt = pd.to_numeric(stops["location_type"], errors="coerce").fillna(0).astype(int)
    LT = {0: "stop_or_platform", 1: "station", 2: "entrance_exit", 3: "generic_node", 4: "boarding_area"}
    stops["loc_type"] = lt.map(LT).fillna("other")
    lt_dist = stops["loc_type"].value_counts()
    p.observe("location_type_distribution", lt_dist)

    has_parent = stops["parent_station"].notna() & stops["parent_station"].astype(str).str.strip().ne("")
    n_parented = int(has_parent.sum())
    p.observe("stops_with_parent_station", n_parented)

    entrances = int((stops["loc_type"] == "entrance_exit").sum())
    p.observe("station_entrances_modelled", entrances)
    p.check("station_entrances", "pass" if entrances else "warn",
            f"{entrances:,} station entrances are modelled as location_type 2 — an entrance is where "
            "a step-free path actually begins" if entrances else
            "no station entrances are modelled (location_type 2 is absent), so the feed locates stops "
            "but not the doors used to reach them",
            entrances)
    if not entrances:
        p.limitation("DS-03 does not model station entrances, so the entry point of a station is not published and cannot be routed to.")

    still_unknown = int((has_parent & stops["wb"].eq("0_no_information")).sum())
    p.observe("parented_stops_still_unknown_after_inheritance", still_unknown)
    p.check("parent_inheritance_residual", "info",
            f"{still_unknown:,} child stops still have no wheelchair_boarding value after inheritance, "
            "because their parent station publishes none either",
            still_unknown)
    display(lt_dist)

loc_type
stop_or_platform    29438
generic_node         1340
entrance_exit         856
station               346
Name: count, dtype: int64

## Pathways and levels — station interiors

This is the part that decides whether a lift is an object the data knows about.

`pathways.pathway_mode`:

| value | meaning |
|---|---|
| 1 | walkway |
| 2 | stairs |
| 3 | moving sidewalk |
| 4 | escalator |
| 5 | **elevator** |
| 6 | fare gate |
| 7 | exit gate |

An elevator row is a named, identified object with endpoints. Without one, there is
nothing for a downstream disruption feed to point at.

In [13]:
PATHWAY_MODES = {1: "walkway", 2: "stairs", 3: "moving_sidewalk", 4: "escalator",
                 5: "elevator", 6: "fare_gate", 7: "exit_gate"}

path_frames, level_frames = [], []
for n, meta in structure.items():
    pw = read_inner(n, "pathways.txt")
    if pw is not None and len(pw):
        pw = pw.copy(); pw["mode"] = meta["mode"]; path_frames.append(pw)
    lv = read_inner(n, "levels.txt")
    if lv is not None and len(lv):
        lv = lv.copy(); lv["mode"] = meta["mode"]; level_frames.append(lv)

pathways = pd.concat(path_frames, ignore_index=True) if path_frames else pd.DataFrame()
levels = pd.concat(level_frames, ignore_index=True) if level_frames else pd.DataFrame()

p.observe("pathways_rows", int(len(pathways)))
p.observe("levels_rows", int(len(levels)))
print(f"pathways: {len(pathways):,} rows across {pathways['mode'].nunique() if len(pathways) else 0} modes")
print(f"levels:   {len(levels):,} rows across {levels['mode'].nunique() if len(levels) else 0} modes")

pathways: 5,109 rows across 3 modes
levels:   22 rows across 8 modes


In [14]:
if not len(pathways):
    p.check("pathways_populated", "warn",
            "pathways.txt is present but empty in every mode feed — station interiors are not published, "
            "so lifts, stairs and step-free routes inside stations have no representation in DS-03",
            0)
    p.limitation("DS-03 publishes no station interior. Lifts, stairs and the step-free path from entrance to platform are not represented, so the entry link cannot be answered for stations from this source.")
    lift_rows = 0
else:
    pathways["pw_mode"] = (pd.to_numeric(pathways["pathway_mode"], errors="coerce")
                             .map(PATHWAY_MODES).fillna("other"))
    pw_dist = pathways.groupby(["mode", "pw_mode"]).size().unstack(fill_value=0)
    p.observe("pathway_mode_by_feed", pw_dist)

    lifts = pathways[pathways["pw_mode"] == "elevator"]
    lift_rows = int(len(lifts))
    p.observe("elevator_pathways", lift_rows)
    p.observe("elevator_pathways_by_mode", lifts["mode"].value_counts())
    p.observe("stations_with_an_elevator",
              int(pathways.loc[pathways["pw_mode"] == "elevator", "from_stop_id"].nunique()))

    p.check("pathways_populated", "pass", f"{len(pathways):,} pathway rows are published", int(len(pathways)))
    p.check("lifts_addressable", "pass" if lift_rows else "warn",
            f"{lift_rows:,} lifts are published as identified pathway rows with endpoints and a "
            f"pathway_id — a lift is an addressable object in this feed" if lift_rows else
            "pathways.txt is populated but contains no pathway_mode 5 rows, so no lift is published "
            "as an identified object",
            lift_rows)
    display(pw_dist)
    display(lifts.head(5))

pw_mode,elevator,escalator,exit_gate,fare_gate,stairs,walkway
mode,,,,,,
Interstate train,14,15,8,14,10,243
Metropolitan train,299,120,77,106,454,2797
Regional train,69,43,24,34,96,686


,pathway_id,from_stop_id,to_stop_id,pathway_mode,is_bidirectional,traversal_time,mode,pw_mode
83,vic:rail:CLA_LI1_22249_elevator_1,vic:rail:CLA_LI1,22249,5,1,60,Regional train,elevator
84,vic:rail:CLA_LI1_13718_elevator_1,vic:rail:CLA_LI1,13718,5,1,60,Regional train,elevator
85,vic:rail:CLA_LI1_13719_elevator_1,vic:rail:CLA_LI1,13719,5,1,60,Regional train,elevator
133,vic:rail:DNG_LI1_12187_elevator_1,vic:rail:DNG_LI1,12187,5,1,60,Regional train,elevator
134,vic:rail:DNG_LI1_12188_elevator_1,vic:rail:DNG_LI1,12188,5,1,60,Regional train,elevator


In [15]:
# The step-free question inside a station: can a route avoid stairs?
if len(pathways):
    stair_like = pathways["pw_mode"].isin(["stairs"])
    stepfree = pathways["pw_mode"].isin(["walkway", "elevator", "moving_sidewalk"])
    detail = {
        "stair_pathways": int(stair_like.sum()),
        "step_free_pathways": int(stepfree.sum()),
        "escalator_pathways": int((pathways["pw_mode"] == "escalator").sum()),
        "max_slope_populated": int(pd.to_numeric(pathways.get("max_slope"), errors="coerce").notna().sum())
            if "max_slope" in pathways.columns else 0,
        "min_width_populated": int(pd.to_numeric(pathways.get("min_width"), errors="coerce").notna().sum())
            if "min_width" in pathways.columns else 0,
    }
    p.observe("pathway_step_free_detail", detail)
    p.check("step_free_routing_inside_stations", "info",
            f"{detail['step_free_pathways']:,} step-free pathway segments against "
            f"{detail['stair_pathways']:,} stair segments; max_slope is populated on "
            f"{detail['max_slope_populated']:,} rows and min_width on {detail['min_width_populated']:,}",
            detail)
    p.contract("Treat escalators as neither step-free nor stairs. An escalator is not usable in a wheelchair and is not a stair to be avoided; it is a separate state.")
    display(pd.Series(detail))

stair_pathways          560
step_free_pathways     4108
escalator_pathways      178
max_slope_populated       0
min_width_populated       0
dtype: int64

## Trips and `wheelchair_accessible`

A stop being accessible is not the same as the service calling at it being accessible.

In [16]:
WA_LABELS = {0: "0_no_information", 1: "1_at_least_one_wheelchair", 2: "2_no_wheelchair"}

trip_frames = []
for n, meta in structure.items():
    t = read_inner(n, "trips.txt",
                   usecols=lambda c: c in {"trip_id", "route_id", "service_id", "wheelchair_accessible"})
    if t is None:
        continue
    t = t.copy(); t["mode"] = meta["mode"]; trip_frames.append(t)
trips = pd.concat(trip_frames, ignore_index=True)
p.observe("trips_total", int(len(trips)))

if "wheelchair_accessible" not in trips.columns:
    p.check("wheelchair_accessible_present", "warn",
            "trips.txt does not carry wheelchair_accessible — vehicle accessibility is not published", None)
    p.limitation("DS-03 does not publish trip-level wheelchair accessibility, so an accessible stop cannot be qualified by whether the service calling there is accessible.")
else:
    trips["wa"] = (pd.to_numeric(trips["wheelchair_accessible"], errors="coerce")
                     .fillna(0).astype(int).map(WA_LABELS).fillna("other"))
    tdist = trips["wa"].value_counts()
    tmode = trips.groupby(["mode", "wa"]).size().unstack(fill_value=0)
    unknown_trips = round(100 * tdist.get("0_no_information", 0) / len(trips), 1)
    p.observe("wheelchair_accessible_distribution", tdist)
    p.observe("wheelchair_accessible_by_mode", tmode)
    p.observe("pct_trips_no_accessibility_information", unknown_trips)
    p.check("wheelchair_accessible_present", "warn" if unknown_trips > 20 else "pass",
            f"trips.txt carries wheelchair_accessible; {unknown_trips}% of trips publish no value",
            tdist)
    display(tdist); display(tmode)

wa
1_at_least_one_wheelchair    256888
2_no_wheelchair               22467
Name: count, dtype: int64

wa,1_at_least_one_wheelchair,2_no_wheelchair
mode,,
Interstate train,0,29
Metropolitan bus,183367,1670
Metropolitan train,40314,0
Metropolitan tram,12147,15949
Regional bus,2492,1483
Regional coach,6758,1500
Regional train,11810,19
SkyBus,0,1817


## Feed currency

GTFS carries its own validity window. A feed whose calendar has expired is stale
regardless of when it was fetched, and the fetch manifest cannot see that.

In [17]:
currency = []
for n, meta in structure.items():
    row = {"mode": meta["mode"], "currency_source": None}
    fi = read_inner(n, "feed_info.txt")
    if fi is not None and len(fi):
        row["currency_source"] = "feed_info.txt"
        for col in ("feed_start_date", "feed_end_date", "feed_version"):
            if col in fi.columns:
                row[col] = fi[col].iloc[0]
    cal = read_inner(n, "calendar.txt")
    if cal is not None and len(cal):
        row["calendar_start"] = int(pd.to_numeric(cal["start_date"], errors="coerce").min())
        row["calendar_end"] = int(pd.to_numeric(cal["end_date"], errors="coerce").max())
        row["currency_source"] = row["currency_source"] or "calendar.txt"
    currency.append(row)
cur = pd.DataFrame(currency)
p.observe("feed_currency", cur)

today = int(pd.Timestamp.today().strftime("%Y%m%d"))
end = cur.get("calendar_end", pd.Series(dtype=float)).fillna(0)
expired = cur[end < today]
p.check("feed_validity_window", "warn" if len(expired) else "pass",
        f"{len(expired)} mode feeds have a calendar ending before today ({today}): "
        f"{expired['mode'].tolist()}" if len(expired)
        else f"every mode feed's calendar covers today ({today})",
        cur.to_dict("records"))
p.contract("Derive the DS-03 stale flag from the calendar end date, not from the fetch timestamp. A freshly fetched feed with an expired calendar is stale.")
cur

,mode,currency_source,calendar_start,calendar_end
0,Regional train,calendar.txt,20260827,20261129
1,Metropolitan train,calendar.txt,20260827,20261129
2,Metropolitan tram,calendar.txt,20260827,20261129
3,Metropolitan bus,calendar.txt,20260827,20261129
4,Regional coach,calendar.txt,20260827,20261129
5,Regional bus,calendar.txt,20260827,20261129
6,Interstate train,calendar.txt,20260827,20261129
7,SkyBus,calendar.txt,20260827,20261129


In [18]:
p.save()
for z in _cache.values():
    z.close()
outer.close()

DS-03 — PTV GTFS Schedule
  object   gtfs.zip  (288,398,506 bytes)
  dt       2026-08-31
  sha256   d492e2bc8def586d6b018213c9dea6bc2082649e27e36e5d46b7355282b2c0e7
  manifest hash matches

  Checks (WARN overall)
    [PASS] raw_integrity: SHA-256 of the profiled object is d492e2bc8def586d6b018213c9dea6bc2082649e27e36e5d46b7355282b2c0e7
    [PASS] feed_structure: 8 per-mode google_transit.zip members found inside the outer archive
    [PASS] mode_numbering: every mode directory maps to a documented PTV mode
    [PASS] gtfs_required_files: every mode feed contains the required GTFS files
    [WARN] feed_info_present: 8 mode feeds ship no feed_info.txt, so the feed does not state its own validity window — currency has to be derived from calendar.txt instead
    [WARN] stop_id_uniqueness: 965 stop_id values appear in more than one mode feed — the load key must be (mode_id, stop_id), not stop_id alone
    [PASS] wheelchair_boarding_present: stops.txt carries wheelchair_boarding with an exp